In [ ]:
import pandas as pd
import numpy as np
import sys
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, LSTM, Reshape
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

sys.path.append(os.path.abspath("../Code"))
import vectorize

import gensim.downloader
from gensim.models.fasttext import load_facebook_model

In [ ]:
DATA_PATHS = [
    '../Data/raw/tcga_simple_train.csv',
    '../Data/interim/tcga_simple_train_preprocessed_0.csv', 
    '../Data/interim/tcga_simple_train_preprocessed_1.csv',
    '../Data/interim/tcga_simple_train_preprocessed_2.csv',
    '../Data/interim/tcga_simple_train_preprocessed_3.csv',
    '../Data/interim/tcga_simple_train_preprocessed_4.csv',
    '../Data/interim/tcga_simple_train_preprocessed_3_augmented.csv',
]
RESULTS_PATH = '../Results/Metrics/lstm.csv'
MAX_LEN = 512
vectorizadores = ["Binaria", "Frecuencia", "Tfidf", "Word2Vec", "FastText"]

In [3]:
w2v_model = gensim.downloader.load('word2vec-google-news-300')
fasttext_model = load_facebook_model("../Models/wiki.simple.bin")

In [4]:
def texto_a_secuencia(serie_texto, mapping, max_len):
    sequencias = []
    for texto in serie_texto:
        seq = [mapping[w] for w in str(texto).split() if w in mapping]
        sequencias.append(seq)
    return pad_sequences(sequencias, maxlen=max_len)

def build_lstm_model(input_type, vocab_size=None, input_dim=None):
    model = Sequential()
    
    if input_type == 'secuencia':
        model.add(Embedding(input_dim=vocab_size + 1, output_dim=16, input_length=MAX_LEN))
    else:
        model.add(Reshape((1, input_dim), input_shape=(input_dim,)))
        
    model.add(LSTM(units=8, activation='tanh', dropout=0.6, recurrent_dropout=0))
    model.add(Dense(4, activation='softmax'))
    
    model.compile(optimizer="adam", loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

resultados = []

In [8]:
for data_path in DATA_PATHS:
    dataset_name = os.path.basename(data_path)
    print(f"PROCESANDO DATASET: {dataset_name}\n")    
    df = pd.read_csv(data_path)
        
    X_train_full, X_test, y_train_full, y_test = train_test_split(df['text'], df['t'], test_size=0.2, random_state=23)
    X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=33)
    
    le = LabelEncoder()
    y_train_num = le.fit_transform(y_train)
    y_val_num = le.transform(y_val)
    y_test_num = le.transform(y_test)
    
    for nombre_vec in vectorizadores:
        print(f" -> Evaluando LSTM con vectorización: {nombre_vec}...")
        
        try:
            # 1. Preparación de Entradas
            if nombre_vec in ["Binaria", "Frecuencia", "Tfidf"]:

                if nombre_vec == "Binaria":
                    _, v = vectorize.vectorizacionBinaria(X_train)
                elif nombre_vec == "Frecuencia":
                    _, v = vectorize.vectorizacionFreq(X_train)
                else:
                    _, v = vectorize.vectorizacionTfidf(X_train)
                    
                vocab_map = v.vocabulary_
                index_mapping = {word: (idx + 1) for word, idx in vocab_map.items()}
                
                X_train_nn = texto_a_secuencia(X_train, index_mapping, MAX_LEN)
                X_val_nn = texto_a_secuencia(X_val, index_mapping, MAX_LEN)
                X_test_nn = texto_a_secuencia(X_test, index_mapping, MAX_LEN)
                
                num_words = len(v.get_feature_names_out()) + 1
                model = build_lstm_model('secuencia', vocab_size=num_words)

            elif nombre_vec == "Word2Vec":
                X_train_nn = vectorize.vectorizacionEmbeddings(X_train, w2v_model)
                X_val_nn = vectorize.vectorizacionEmbeddings(X_val, w2v_model)
                X_test_nn = vectorize.vectorizacionEmbeddings(X_test, w2v_model)
                
                input_dim = X_train_nn.shape[2] if len(X_train_nn.shape) > 2 else X_train_nn.shape[1]
                model = build_lstm_model('embedding', input_dim=input_dim)

            elif nombre_vec == "FastText":
                X_train_nn = vectorize.vectorizacionEmbeddings(X_train, fasttext_model)
                X_val_nn = vectorize.vectorizacionEmbeddings(X_val, fasttext_model)
                X_test_nn = vectorize.vectorizacionEmbeddings(X_test, fasttext_model)
                
                input_dim = X_train_nn.shape[2] if len(X_train_nn.shape) > 2 else X_train_nn.shape[1]
                model = build_lstm_model('embedding', input_dim=input_dim)

            early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
            
            history = model.fit(
                X_train_nn, y_train_num,
                epochs=30,
                batch_size=64,
                validation_data=(X_val_nn, y_val_num),
                callbacks=[early_stop],
                verbose=0 
            )
            
            # 3. Evaluación
            predicciones_prob = model.predict(X_test_nn, verbose=0)
            clases_predichas = np.argmax(predicciones_prob, axis=1)
            macro_f1 = f1_score(y_test_num, clases_predichas, average='macro')
            
            # 4. Guardar Resultado con Dataset ID
            resultados.append({
                "Dataset": dataset_name,
                "Vectorizacion": nombre_vec,
                "Macro_F1": round(macro_f1, 4)
            })
            
            print(f"F1 Macro: {macro_f1:.4f}")
            
        finally:
            tf.keras.backend.clear_session()

# 5. Exportar a CSV
df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv(RESULTS_PATH, index=False)
print(df_resultados.head())

PROCESANDO DATASET: tcga_simple_train_preprocessed_3_augmented.csv

 -> Evaluando LSTM con vectorización: Binaria...


c:\Users\tcspo\Desktop\Codigo_Clases\PLN1\practica\OncoNLP\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


F1 Macro: 0.6188
 -> Evaluando LSTM con vectorización: Frecuencia...


c:\Users\tcspo\Desktop\Codigo_Clases\PLN1\practica\OncoNLP\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


F1 Macro: 0.5785
 -> Evaluando LSTM con vectorización: Tfidf...


...

F1 Macro: 0.5629
 -> Evaluando LSTM con vectorización: Word2Vec...


c:\Users\tcspo\Desktop\Codigo_Clases\PLN1\practica\OncoNLP\.venv\Lib\site-packages\keras\src\layers\reshaping\reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


F1 Macro: 0.5686
 -> Evaluando LSTM con vectorización: FastText...


c:\Users\tcspo\Desktop\Codigo_Clases\PLN1\practica\OncoNLP\.venv\Lib\site-packages\keras\src\layers\reshaping\reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


F1 Macro: 0.5507
                 Dataset Vectorizacion  Macro_F1
0  tcga_simple_train.csv       Binaria    0.4124
1  tcga_simple_train.csv    Frecuencia    0.3335
2  tcga_simple_train.csv         Tfidf    0.2471
3  tcga_simple_train.csv      Word2Vec    0.3921
4  tcga_simple_train.csv      FastText    0.3426
